In [ ]:
pip install crewai

In [1]:
import os
from getpass import getpass
from crewai import LLM,Agent,Task

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ").strip()
llm = LLM(model="gpt-4o",temperature=0,)
print("OpenAI model configured: gpt-4o")

Enter your OpenAI API key:  ········


OpenAI model configured: gpt-4o


# **Create Agent 1 — HR Policy Agent**

In [2]:
hr_policy_agent = Agent(
    role="HR Policy Specialist",
    goal=(
        "Analyze employee HR policies and determine "
        "the rules applicable to the employee's situation."
    ),
    backstory=(
        "You are an HR policy specialist who understands "
        "company policies related to notice periods, leave, "
        "resignation and employee benefits."
    ),
    llm=llm,
    verbose=True
)

In [3]:
hr_policy_task = Task(
    description=(
        "Analyze the employee's request: {employee_request}. "
        "Identify the applicable notice period, leave policy "
        "and resignation rules relevant to the request."
    ),
    expected_output=(
        "A clear summary of the applicable HR policies, "
        "including notice period, leave rules and "
        "resignation requirements."
    ),
    agent=hr_policy_agent
)

# **Create Agent 2 — Payroll Agent**

In [4]:
payroll_agent = Agent(
    role="Payroll Specialist",
    goal=(
        "Calculate the employee's final financial settlement "
        "based on salary, leave balance, deductions and "
        "other applicable payments."
    ),
    backstory=(
        "You are a payroll specialist experienced in "
        "employee settlements, salary calculations, "
        "leave encashment and deductions."
    ),
    llm=llm,
    verbose=True
)

In [5]:
payroll_task = Task(
    description=(
        "Analyze the employee's request: {employee_request}. "
        "Calculate the employee's final settlement using "
        "salary, leave balance, applicable encashment, "
        "deductions and pending reimbursements, if requested."
    ),
    expected_output=(
        "A clear breakdown of salary, leave encashment, "
        "deductions, reimbursements and final settlement, "
        "where relevant and supported by the available information."
    ),
    agent=payroll_agent
)

# **Create Agent 3 — Compliance Agent**

In [6]:
compliance_agent = Agent(
    role="HR Compliance Specialist",
    goal=(
        "Identify all employee exit procedures, clearances "
        "and required documents before the employee's "
        "last working day."
    ),
    backstory=(
        "You are an HR compliance specialist responsible "
        "for employee exit procedures, clearances and "
        "required documentation."
    ),
    llm=llm,
    verbose=True
)

In [7]:
compliance_task = Task(
    description=(
        "Analyze the employee's request: {employee_request}. "
        "Identify relevant exit procedures, clearances "
        "and required documents before the employee's "
        "last working day, if requested."
    ),
    expected_output=(
        "A complete employee exit checklist containing "
        "relevant procedures, clearances and documents."
    ),
    agent=compliance_agent
)

# **Create Agent 4 — Documentation Agent**

In [8]:
documentation_agent = Agent(
    role="HR Documentation Specialist",
    goal=(
        "Prepare professional resignation and employee "
        "handover documents using the information provided "
        "by the other HR specialists."
    ),
    backstory=(
        "You are an HR documentation specialist who creates "
        "clear and professional resignation letters, "
        "handover checklists and HR communications."
    ),
    llm=llm,
    verbose=True
)

In [9]:
documentation_task = Task(
    description=(
        "Analyze the employee's request: {employee_request}. "
        "Prepare the relevant resignation letter, "
        "handover checklist or HR communication requested "
        "by the employee."
    ),
    expected_output=(
        "A professional resignation letter, handover "
        "checklist or HR communication, as requested."
    ),
    agent=documentation_agent
)

# **Main Orchestrator AGENT — The HR Manager**

In [10]:
hr_manager = Agent(
    role="HR Manager / Orchestrator",
    goal=(
        "Coordinate specialized HR agents to resolve the "
        "employee's request and produce a complete final response."
    ),
    backstory=(
        "You are an experienced HR manager responsible for "
        "coordinating HR policy, payroll, compliance and "
        "documentation specialists."
    ),
    llm=llm,
    verbose=True
)

In [11]:
final_hr_task = Task(
    description=(
        "Resolve the employee's request: {employee_request}. "
        "Coordinate the relevant HR policy, payroll, "
        "compliance and documentation specialists as needed. "
        "Provide a final response focused on what the employee requested."
    ),
    expected_output=(
        "A clear, relevant response to the employee's request, "
        "including only the applicable HR information."
    ),
    agent=hr_manager
)

# **Create the Agent Crew**

In [12]:
from crewai import Crew, Process

def create_hr_crew(user_input):

    request = user_input.lower()

    if any(word in request for word in [
        "notice period", "leave policy", "hr policy", "benefits"
    ]):
        selected_agent = hr_policy_agent
        selected_task = hr_policy_task

    elif any(word in request for word in [
        "salary", "payroll", "final settlement", "encashment", "deduction"
    ]):
        selected_agent = payroll_agent
        selected_task = payroll_task

    elif any(word in request for word in [
        "clearance", "exit procedure", "company property", "last working day"
    ]):
        selected_agent = compliance_agent
        selected_task = compliance_task

    elif any(word in request for word in [
        "resignation letter", "write a letter", "handover document", "draft an email"
    ]):
        selected_agent = documentation_agent
        selected_task = documentation_task

    else:
        selected_agent = hr_policy_agent
        selected_task = hr_policy_task

    hr_crew = Crew(
        agents=[selected_agent],
        tasks=[selected_task],
        process=Process.sequential,
        verbose=True
    )

    return hr_crew

In [13]:
user_input = input("Enter your HR question: ")

hr_crew = create_hr_crew(user_input)

result = await hr_crew.kickoff_async(
    inputs={"employee_request": user_input}
)

print("\nFinal HR Response:")
print(result)

Enter your HR question:  What's nthe sick leave policy and draft a sick leave email


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3e7830f6-f65a-4976-a0e2-997cf94a6990                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the employee's request: What's nthe sick leave policy and draft a sick leave email. Identify     │
│  the applicable notice period, leave policy and resignation rules relevant to the request.                      │
│  ID: e00fc871-5bdc-4a95-aecc-7c5b34a040d0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: HR Policy Specialist                                                                                    │
│                                                                                                                 │
│  Task: Analyze the employee's request: What's nthe sick leave policy and draft a sick leave email. Identify     │
│  the applicable notice period, leave policy and resignation rules relevant to the request.                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: HR Policy Specialist                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To effectively address the employee's request regarding sick leave, it is essential to outline the relevant    │
│  HR policies, including the sick leave policy, notice period, leave rules, and resignation requirements. Below  │
│  is a comprehensive analysis and a draft email for requesting sick leave:                                       │
│                                                                                                                 │
│  ### Sick Leave Policy                                                                                          │
│  1. **Eligibility**: Employees are eligible for sick leave from the start of their employment.                  │
│  2. **Entitlement**: Employees are entitled to a certain number of paid sick leave days per year, typically     │
│  ranging from 5 to 10 days, depending on the company's policy.                                                  │
│  3. **Documentation**: A medical certificate may be required if the sick leave extends beyond a specified       │
│  number of days, usually three consecutive days.                                                                │
│  4. **Notification**: Employees should notify their supervisor or HR as soon as possible, ideally before the    │
│  start of their workday, if they are unable to attend work due to illness.                                      │
│                                                                                                                 │
│  ### Notice Period                                                                                              │
│  - **Standard Notice**: The standard notice period for resignation is typically two weeks, but this can vary    │
│  based on the employee's contract or company policy.                                                            │
│  - **During Sick Leave**: If an employee is on sick leave, the notice period may still apply, but it is         │
│  advisable to check the specific terms in the employment contract.                                              │
│                                                                                                                 │
│  ### Leave Policy                                                                                               │
│  - **Accrual**: Sick leave is usually accrued annually and may not carry over to the next year unless           │
│  specified by the company policy.                                                                               │
│  - **Concurrent Leave**: Sick leave cannot be taken concurrently with other types of leave, such as annual      │
│  leave, unless approved by HR.                                                                                  │
│                                                                                                                 │
│  ### Resignation Requirements                                                                                   │
│  - **Formal Notice**: Employees must submit a formal resignation letter, adhering to the notice period          │
│  specified in their contract.                                                                                   │
│  - **Exit Process**: Employees should complete any exit procedures, including returning company property and    │
│  completing exit interviews, if applicable.            

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyze the employee's request: What's nthe sick leave policy and draft a sick leave email. Identify     │
│  the applicable notice period, leave policy and resignation rules relevant to the request.                      │
│  Agent: HR Policy Specialist                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Final HR Response:
To effectively address the employee's request regarding sick leave, it is essential to outline the relevant HR policies, including the sick leave policy, notice period, leave rules, and resignation requirements. Below is a comprehensive analysis and a draft email for requesting sick leave:

### Sick Leave Policy
1. **Eligibility**: Employees are eligible for sick leave from the start of their employment.
2. **Entitlement**: Employees are entitled to a certain number of paid sick leave days per year, typically ranging from 5 to 10 days, depending on the company's policy.
3. **Documentation**: A medical certificate may be required if the sick leave extends beyond a specified number of days, usually three consecutive days.
4. **Notification**: Employees should notify their supervisor or HR as soon as possible, ideally before the start of their workday, if they are unable to attend work due to illness.

### Notice Period
- **Standard Notice**: The standard notice period

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3e7830f6-f65a-4976-a0e2-997cf94a6990                                                                       │
│  Final Output: To effectively address the employee's request regarding sick leave, it is essential to outline   │
│  the relevant HR policies, including the sick leave policy, notice period, leave rules, and resignation         │
│  requirements. Below is a comprehensive analysis and a draft email for requesting sick leave:                   │
│                                                                                                                 │
│  ### Sick Leave Policy                                                                                          │
│  1. **Eligibility**: Employees are eligible for sick leave from the start of their employment.                  │
│  2. **Entitlement**: Employees are entitled to a certain number of paid sick leave days per year, typically     │
│  ranging from 5 to 10 days, depending on the company's policy.                                                  │
│  3. **Documentation**: A medical certificate may be required if the sick leave extends beyond a specified       │
│  number of days, usually three consecutive days.                                                                │
│  4. **Notification**: Employees should notify their supervisor or HR as soon as possible, ideally before the    │
│  start of their workday, if they are unable to attend work due to illness.                                      │
│                                                                                                                 │
│  ### Notice Period                                                                                              │
│  - **Standard Notice**: The standard notice period for resignation is typically two weeks, but this can vary    │
│  based on the employee's contract or company policy.                                                            │
│  - **During Sick Leave**: If an employee is on sick leave, the notice period may still apply, but it is         │
│  advisable to check the specific terms in the employment contract.                                              │
│                                                                                                                 │
│  ### Leave Policy                                                                                               │
│  - **Accrual**: Sick leave is usually accrued annually and may not carry over to the next year unless           │
│  specified by the company policy.                                                                               │
│  - **Concurrent Leave**: Sick leave cannot be taken concurrently with other types of leave, such as annual      │
│  leave, unless approved by HR.                                                                                  │
│                                                                                                                 │
│  ### Resignation Requirements                                                                                   │
│  - **Formal Notice**: Employees must submit a formal resignation letter, adhering to the notice period          │
│  specified in their contract.                                                                                   │
│  - **Exit Process**: Employees should complete any exit procedures, including returning company property and    │
│  completing exit interviews, if applicable.           

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# **THE END.**